# Learning a CNN for Image Classification (PyTorch)

This is the PyTorch version of the Keras [course notebook](course_notebook.ipynb). The data loading and exploration sections are identical — only the model building, training, and evaluation sections differ.

* Based on [Deep Learning with Python](https://www.manning.com/books/deep-learning-with-python-second-edition?gclid=CjwKCAjw9aiIBhA1EiwAJ_GTSlKgxc4qopKHPsFWryOoTz7fvhvhzYSjEsgQ-bG1R51QSGppISywpBoClcIQAvD_BwE) by Francois Chollett

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
from tensorflow.keras.datasets import fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

Let's create a little Python list so that we can go from numbers to descriptions easily.

In [ ]:
labels = ["T-shirt/top",
          "Trouser",
          "Pullover",
          "Dress",
          "Coat",
          "Sandal",
          "Shirt",
          "Sneaker",
          "Bag",
          "Ankle boot"]

## A Convolutional Neural Network


### Convolutional Layers


We will follow the same sequence of steps as we did above:


*   Data Prep
*   Define Model
*   Set Optimization Parameters
*   Train Model
*   Evaluate Model

### Data Prep

As we did before, let's normalize to the 0-1 range by dividing everything by 255.

In [ ]:
x_train = x_train / 255.0
x_test = x_test / 255.0

In [ ]:
x_train.shape

### Convert to PyTorch tensors and create DataLoader

For CNNs, PyTorch uses **channels-first** format `(N, C, H, W)` — so each image becomes `(1, 28, 28)` instead of Keras' `(28, 28, 1)`. We add the channel dimension when creating the tensors.

We also manually split training data into train (80%) and validation (20%), matching the Keras `validation_split=0.2`.

In [ ]:
# Split training data into train and validation (80/20)
n_val = int(len(x_train) * 0.2)

# Add channel dimension: (N, 28, 28) -> (N, 1, 28, 28) for PyTorch channels-first format
x_val_t = torch.tensor(x_train[:n_val], dtype=torch.float32).unsqueeze(1)
y_val_t = torch.tensor(y_train[:n_val], dtype=torch.long)
x_tr_t = torch.tensor(x_train[n_val:], dtype=torch.float32).unsqueeze(1)
y_tr_t = torch.tensor(y_train[n_val:], dtype=torch.long)
x_test_t = torch.tensor(x_test, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.long)

batch_size = 64
train_loader = DataLoader(TensorDataset(x_tr_t, y_tr_t), batch_size=batch_size, shuffle=True)

print(f"Training:   {x_tr_t.shape}")
print(f"Validation: {x_val_t.shape}")
print(f"Test:       {x_test_t.shape}")

### Define Model

OK, we are ready to create our very first **Convolutional Neural Network (CNN)!**

The architecture mirrors the Keras notebook:
- Conv2d(32, kernel 2x2, relu) → MaxPool2d → Conv2d(32, kernel 2x2, relu) → MaxPool2d → Flatten → Linear(256, relu) → Linear(10)

Note: In PyTorch, `CrossEntropyLoss` applies softmax internally, so we don't add a softmax layer.

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # First convolutional block
            nn.Conv2d(1, 32, kernel_size=2),    # (1, 28, 28) -> (32, 27, 27)
            nn.ReLU(),
            nn.MaxPool2d(2),                     # (32, 27, 27) -> (32, 13, 13)

            # Second convolutional block
            nn.Conv2d(32, 32, kernel_size=2),   # (32, 13, 13) -> (32, 12, 12)
            nn.ReLU(),
            nn.MaxPool2d(2),                     # (32, 12, 12) -> (32, 6, 6)

            # Flatten and dense layers
            nn.Flatten(),                        # (32, 6, 6) -> (1152,)
            nn.Linear(1152, 256),                # (1152,) -> (256,)
            nn.ReLU(),
            nn.Linear(256, 10),                  # (256,) -> (10,)
        )

    def forward(self, x):
        return self.net(x)

model = FashionCNN()

In [ ]:
print(model)

In [ ]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params}")

Let's hand-calculate the number of parameters to verify.

- Conv_1: 1 input channel, 32 filters, 2x2 kernel = 32 * (1 * 2 * 2 + 1) = 160
- Conv_2: 32 input channels, 32 filters, 2x2 kernel = 32 * (32 * 2 * 2 + 1) = 4128
- Linear_1: 1152 * 256 + 256 = 295168
- Linear_2: 256 * 10 + 10 = 2570

In [ ]:
160 + 4128 + 295168 + 2570

### Set Optimization Parameters

In PyTorch, we set up the loss function and optimizer as separate objects.

* **Loss function**: `nn.CrossEntropyLoss()` — combines log-softmax and NLL loss. Equivalent to Keras' `sparse_categorical_crossentropy`.
* **Optimizer**: `torch.optim.Adam` — the same Adam optimizer used in the Keras notebook.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### Train the Model

Unlike Keras where `model.fit()` handles the entire training loop, in PyTorch we write the training loop explicitly. This gives us full control over what happens at each step:

1. **Forward pass**: compute predictions
2. **Compute loss**: compare predictions to targets
3. **Backward pass**: compute gradients
4. **Update weights**: optimizer takes a step

In [ ]:
num_epochs = 10

history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

for epoch in range(num_epochs):
    # --- Training phase ---
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0

    for xb, yb in train_loader:
        optimizer.zero_grad()           # reset gradients
        logits = model(xb)              # forward pass
        loss = criterion(logits, yb)    # compute loss
        loss.backward()                 # backward pass
        optimizer.step()                # update weights

        epoch_loss += loss.item() * len(xb)
        epoch_correct += (logits.argmax(1) == yb).sum().item()
        epoch_total += len(xb)

    train_loss = epoch_loss / epoch_total
    train_acc = epoch_correct / epoch_total

    # --- Validation phase ---
    model.eval()
    with torch.no_grad():
        val_logits = model(x_val_t)
        val_loss = criterion(val_logits, y_val_t).item()
        val_acc = (val_logits.argmax(1) == y_val_t).float().mean().item()

    history["loss"].append(train_loss)
    history["accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - "
          f"val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

As usual, let's plot the loss and accuracy curves

In [ ]:
epochs_range = range(1, num_epochs + 1)
plt.plot(epochs_range, history["loss"], "bo", label="Training loss", markersize=2)
plt.plot(epochs_range, history["val_loss"], "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.clf()
plt.plot(epochs_range, history["accuracy"], "bo", label="Training acc", markersize=2)
plt.plot(epochs_range, history["val_accuracy"], "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

The validation loss/accuracy are flattening out but not increasing yet so we can just go forward with this model.

### Evaluate the Model

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(x_test_t)
    test_loss = criterion(test_logits, y_test_t).item()
    test_acc = (test_logits.argmax(1) == y_test_t).float().mean().item()

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

**Excellent! We got to over 90% accuracy on the test set - nice.**


> Using two **specialized layers - the convolutional and pooling layers -** helped us exceed 90%.


Back to Fashion MNIST. Let's see what the **[state of the art (SOTA)](https://paperswithcode.com/sota/image-classification-on-fashion-mnist)** accuracy is.

It is **96.91%**!

<br>


**Challenge: Can you get to SOTA by playing around with the architecture of the network?** Add more convolutional layers, change the size of each convolutional filter etc.


## Iteratively Improving the Model

> **Practical Tip**: Once any model is built, it is a good idea to look at the predictions on the test set and see what types of examples the model has difficulty predicting. That can often suggest ways in which the model can be improved.

For problems where the input data consists of *images* (like Fashion MNIST), **visualization** can be very helpful.

In [ ]:
# get the predictions
model.eval()
with torch.no_grad():
    y_hat = model(x_test_t).argmax(1).numpy()

# collect examples where the model made a mistake
misses = np.where(y_hat != y_test)

In [ ]:
# Plot a random sample of 25 test images incorrectly classified by the model,
# their predicted labels and ground truth
figure = plt.figure(figsize=(20, 8))
for i, index in enumerate(np.random.choice(misses[0], size=25, replace=False)):
    ax = figure.add_subplot(5, 5, i + 1, xticks=[], yticks=[])
    # Display each image — squeeze out the channel dimension for display
    ax.imshow(x_test_t[index].squeeze(), cmap="gray")
    # Set the title for each image
    ax.set_title("{} ({})".format(labels[y_hat[index]],
                                  labels[y_test[index]]))

You can see that the model seems to be having difficulty distinguishing between *visually similar* categories.

The **confusion matrix** is a good way to get a complete picture of this phenomenon.

In [ ]:
actuals = [labels[i] for i in y_test]
predictions = [labels[i] for i in y_hat]

In [ ]:
df = pd.DataFrame({'Predictions': predictions, 'Actuals': actuals})
pd.crosstab(df.Predictions, df.Actuals)

**Observations**

*   All the off-diagonal numbers represent mistakes made by the model.
*   You can see that the model made the most mistakes for "Shirts". It confused a "Shirt" for a "T-shirt/Top" often. This is understandable since the products are visually similar.


---
**How can we improve the model?**

> **Tip**: Get more data on those categories that the model has difficulty distinguishing between (e.g., Shirts and T-Shirts), enrich the original training dataset with these new examples and re-train the model.


## Conclusion

We have built a Deep Learning model that can classify grayscale images of clothing items with over 90% accuracy!!


In the [next colab](https://colab.research.google.com/drive/18TNYD4_5a2PA3UTPOc4j2RNBFrofuk_T), we will discuss:
* How to work with color images
* A general strategy - **transfer learning** - for solving problems in practice
* Apply the general strategy to build a handbags/shoes classifier with just 100 examples!

**DONE**
